# 04 — Beijing Feature Engineering

Tạo feature cho forecasting t+1; lag/rolling chỉ nhìn quá khứ.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shutil

# ------------------------------------------------------------
# Robust project root finder
# Notebook lives under: Project/notebooks/son_nb/
# ------------------------------------------------------------
def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for p in candidates:
        if (p / "data" / "raw").exists() and (p / "notebooks").exists():
            return p
    raise FileNotFoundError(
        "Không tìm thấy Project root. Hãy chạy notebook bên trong project."
    )

PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MAIN_OUTPUT = PROJECT_ROOT / "outputs"
SON_DIR = PROJECT_ROOT / "notebooks" / "son_nb"
SON_OUTPUT = SON_DIR / "output"

WORK_DATA = SON_OUTPUT / "data"
WORK_FIG = SON_OUTPUT / "figures"
WORK_TABLE = SON_OUTPUT / "tables"
WORK_PRED = SON_OUTPUT / "predictions"
WORK_MODEL = SON_OUTPUT / "models"

for p in [WORK_DATA, WORK_FIG, WORK_TABLE, WORK_PRED, WORK_MODEL]:
    p.mkdir(parents=True, exist_ok=True)

BEIJING_RAW = RAW_DIR / "beijing" / "PRSA_data_2010.1.1-2014.12.31.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("BEIJING_RAW:", BEIJING_RAW)
print("Working output:", SON_OUTPUT)

In [ ]:
df=pd.read_csv(WORK_DATA/'beijing_hourly_clean_working.csv',parse_dates=['datetime']).sort_values('datetime').reset_index(drop=True)
df['target_pm25_t1']=df['pm2.5'].shift(-1)
for lag in [1,2,3,6,12,24,48,72]: df[f'pm25_lag_{lag}']=df['pm2.5'].shift(lag)
for w in [3,6,12,24]:
    s=df['pm2.5'].shift(1)
    df[f'pm25_roll_mean_{w}']=s.rolling(w).mean()
    df[f'pm25_roll_std_{w}']=s.rolling(w).std()
for col in ['DEWP','TEMP','PRES','Iws','Is','Ir']:
    for lag in [1,3,6,12,24]: df[f'{col}_lag_{lag}']=df[col].shift(lag)

In [ ]:
m=pd.get_dummies(df,columns=['cbwd'],prefix='wind',dtype=int)
base=['DEWP','TEMP','PRES','Iws','Is','Ir','hour_sin','hour_cos','month_sin','month_cos','dayofweek']
derived=[c for c in m.columns if c.startswith('pm25_lag_') or c.startswith('pm25_roll_') or any(c.startswith(f'{w}_lag_') for w in ['DEWP','TEMP','PRES','Iws','Is','Ir'])]
wind=[c for c in m.columns if c.startswith('wind_')]
features=base+derived+wind
ready=m[features+['target_pm25_t1','datetime']].dropna().sort_values('datetime').reset_index(drop=True)
print('ready shape:',ready.shape); print('feature count:',len(features))

In [ ]:
train=ready[ready.datetime.dt.year<=2012].copy(); valid=ready[ready.datetime.dt.year==2013].copy(); test=ready[ready.datetime.dt.year==2014].copy()
train.to_csv(WORK_DATA/'beijing_train_working.csv',index=False); valid.to_csv(WORK_DATA/'beijing_valid_working.csv',index=False); test.to_csv(WORK_DATA/'beijing_test_working.csv',index=False)
(WORK_DATA/'beijing_feature_list.txt').write_text('\n'.join(features),encoding='utf-8')
print(train.shape,valid.shape,test.shape)

### Leakage checklist
- Rolling dùng `shift(1)`
- target t+1 không nằm trong feature
- split theo thời gian
- chưa fit imputer/scaler trên toàn bộ dataset